# 10_03 An IT service desk: route the ticket, suggest the article

*📖 Not in the book*

**How to use this notebook:** click a cell and press Shift+Enter to run just that cell.
Before each code cell, jot down what you think it will do. Then run it and see if you were right.

Run the cells in order, from the top. **A cell can use what earlier cells made**, so if you skip ahead you'll get `NameError: name '…' is not defined`. If that happens, run the cells above it, or just click Run All.

> **Where this fits**
>
> ```
> Odyssey text → numbers → model reads → model learns → model writes → MCP tool
> ```
>
> You are here: **after the main course**: rung 2 again, on the tickets of a service desk
>
> Project files this connects to:
>
> - `src/homer_ai/domain.py`
> - `src/homer_ai/retrieval.py`
> - `data/domain/servicedesk/`
> - `scripts/make_domain_data.py`
>
> **This notebook:** Every company has a service desk. How do we send a ticket to the right team and suggest a how-to article with no training, and how does the way we split the data change the score?
>
> For the big picture, what we're building and why, see the [course map](../00_start_here/00_00_course_map.ipynb).

## 1. What you'll learn

- How to suggest a how-to article for a ticket with the BM25 of chapter 9, and how to score it
- How to route a ticket to a team with no training at all: let its nearest solved tickets vote
- Why greetings and sign-offs fool a similarity search, and how idf removes them
- How much a test score depends on the way the data was split
- What a short fine-tuning run adds here, and what it doesn't

## 2. Words you'll meet

| Word | Meaning in plain English | Example |
|---|---|---|
| **triage** | the first sorting of incoming work: what is it, who gets it, how urgent | a ticket goes to the network team |
| **knowledge base** | a collection of how-to articles for known problems; KB for short | `kb-03`: Connect to the VPN from home |
| **routing** | choosing the team or queue a ticket goes to | one of 6 categories |
| **nearest neighbours** | classifying a new item by looking at the labelled items most like it; k-NN for short | the 5 most similar solved tickets vote |
| **boilerplate** | text that is repeated everywhere and says nothing about the case | *Hi team*, *Thanks*, *Sent from my phone* |

## 3. The idea

### ❓ What are we building?

The front door of an IT service desk. Somebody reads each new ticket, drops it in the right team's tray, and staples the how-to article on it if there is one. That reading and sorting is called **triage**, and it's the same boring job at every company.

We do both halves with tools you already have:

| Half | In this notebook | Built on |
|---|---|---|
| staple the article on | `suggest_articles`: BM25 over 10 knowledge-base articles | [09_01](../09_advanced_grounded_answers/09_01_chunking_and_bm25.ipynb) |
| pick the tray | a router that asks which solved tickets look like this one | BM25 again, used as a classifier |

Everything here is invented: the firm, its products, its customers and their words ([data/domain/README.md](../../data/domain/README.md)).

Nothing here is risky, and that's the point of the example. A ticket in the wrong tray costs somebody a few minutes. So it's a good place for a first project, and a good place to learn what fools a test.

## 4. Step by step

Each step is one question. Read it, guess what the code will do, run it, then look at what you got.

### ❓ Step 1: what does the data look like?

200 tickets and 10 articles. A ticket has its text, the `category` (the team that solved it) and `kb`, the article that solves it, if any.

👉 **Predict first:** for how many of the 200 tickets is there no article at all?

In [ ]:
from homer_ai.domain import load_kb, load_tickets

articles = load_kb()
train, test = load_tickets("train"), load_tickets("test")
print(
    len(articles),
    "articles,",
    len(train),
    "solved tickets to learn from,",
    len(test),
    "test tickets",
)
print()
print(f"{'category':9} {'article':8} ticket")
for ticket in train[:6]:
    print(f"{ticket['category']:9} {ticket['kb'] or '-':8} {ticket['text']}")
print()
for article in articles[:3]:
    print(f"{article.id}  {article.category:9} {article.title}")
print("…")
everything = train + test
print(
    f"\ntickets with no article: {sum(t['kb'] is None for t in everything)} "
    f"of {len(everything)}"
)

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

Short messages, written the way people write to IT: often no capitals, no full stop, *Thanks* at the end. Six categories: `access`, `email`, `hardware`, `network`, `security`, `software`.

78 of the 200 have no article. A lost laptop or a cracked screen isn't solved by reading something. For those, the right result is *no article, a person picks this up*, and we'll see whether our search can say that.

The tickets come from the same script as the emails of 10_02, and they're split the same way: by wording. Each kind of problem has 8 ways of saying it, 6 for training and 2 kept for the test.

</details>

### ❓ Step 2: how do I suggest an article?

It's the search of chapter 9 with the ticket as the question. `suggest_articles` puts the 10 articles (title and text) in a `BM25` index and returns the best matches.

Two tickets, both about articles that exist:

```
VPN won't connect from home                            → kb-03, Connect to the VPN from home
second step of login fails since I changed handset     → kb-02, Set up multi-factor authentication on a new phone
```

👉 **Predict first:** which of the two will the search get right?

In [ ]:
from homer_ai.domain import suggest_articles

for text in (
    "VPN won't connect from home",
    "second step of login fails since I changed handset",
):
    print(text)
    for article, score in suggest_articles(text, k=3):
        print(f"   {score:5.1f}  {article.id}  {article.title}")
    print()

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The first is easy: `kb-03` wins by a mile, because ticket and article share *VPN*, *connect* and *home*.

The second fails. The right article, `kb-02`, isn't among the three. The user wrote *second step of login* and *handset*. The article says *multi-factor authentication* and *phone*. Same thing, no shared word, and a word search has nothing to hold on to.

People who write tickets don't use the words of people who write articles. That's the main problem of every help-desk search.

</details>

### ❓ Step 3: how often is the right article among the suggestions?

We count, on the test tickets that have an article. The score is **recall@k** from [09_02](../09_advanced_grounded_answers/09_02_search_and_scores.ipynb): the share of tickets whose right article is among the first k.

We look at k = 3, because that's what a service-desk screen would show: three suggestions, and the agent picks.

👉 **Predict first:** for how many of the 26 is the right article the first suggestion?

In [ ]:
from homer_ai.retrieval import retrieval_scores

with_article = [t for t in test if t["kb"]]
ranks = []
for ticket in with_article:
    found = [article.id for article, _score in suggest_articles(ticket["text"], k=10)]
    ranks.append(found.index(ticket["kb"]) + 1 if ticket["kb"] in found else None)

scores = retrieval_scores(ranks, ks=(1, 3))
print(f"{len(with_article)} test tickets with an article")
print(f"recall@1 {scores['recall@1']:.3f}   recall@3 {scores['recall@3']:.3f}")
print("\nright article not in the first 3:")
for ticket, rank in zip(with_article, ranks, strict=True):
    if rank is None or rank > 3:
        print(f"   rank {rank}   {ticket['kb']}  {ticket['text']}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

First suggestion right for 17 of 26 (0.654), and among the first three for 22 of 26 (0.846).

The four misses are two wordings, each seen twice: the *handset* ticket of step 2, and *need the support inbox to appear on my laptop*, where the article talks about a *shared mailbox*.

The usual fix isn't a cleverer search. It's to add the users' own words to the article: a line like *also: two-step sign in, code app, new handset*. Solved tickets are where you find those words.

</details>

### ❓ Your turn: what's recall@5?

A wider screen could show five suggestions. Work out recall@5 from `ranks` yourself: the share of the 26 tickets whose right article is among the first 5.

`ranks` holds one entry per ticket: the place of the right article (1 is the top), or `None` if it wasn't among the 10 we asked for.

👉 **Predict first:** recall@3 was 0.846. Can recall@5 be lower than that?

Replace each `None`, run the cell, then run the check cell under it.

In [ ]:
# 👉 do: the share of tickets whose rank is 5 or better
# 💡 hint: a rank can be None, and None <= 5 is an error, so test for None first
recall_at_5 = None

In [ ]:
from homer_ai.lessons import check

check(
    "recall@5",
    recall_at_5,
    retrieval_scores(ranks, ks=(5,))["recall@5"],
    coach="Divide by ALL the tickets, len(ranks), also the ones whose article "
    "wasn't found: a miss has to count against the search. If you divided by "
    "the found ones only, the score comes out too high.",
)

<details><summary>💡 Hint</summary>

Count the ranks that pass the test, with `sum(... for rank in ranks)`, then divide by `len(ranks)`.

</details>

<details><summary>✅ Answer</summary>

```python
recall_at_5 = sum(rank is not None and rank <= 5 for rank in ranks) / len(ranks)
```

It can't be lower than recall@3: every ticket that's right in the first 3 is also right in the first 5. Here it comes out at 0.846 again, 22 of 26. Two more suggestions on the screen found nothing new, so the four misses need the fix from above: the users' own words in the articles.

</details>

### ❓ Step 4: can the search tell me when there's no article?

That would be useful: 78 tickets have none, and three confident suggestions for a stolen phone are worse than no suggestion.

The natural idea is a floor. If even the best article scores low, say *no article*. Let's see whether the scores allow it.

👉 **Predict first:** will tickets with an article score clearly higher than tickets without one?

In [ ]:
def best_score(ticket):
    found = suggest_articles(ticket["text"], k=1)
    return found[0][1] if found else 0.0


has_one = sorted(best_score(t) for t in test if t["kb"])
has_none = sorted(best_score(t) for t in test if not t["kb"])
print(f"{'best score, tickets':22} {'lowest':>7} {'middle':>7} {'highest':>8}")
for name, values in (("with an article", has_one), ("with no article", has_none)):
    print(
        f"{name:22} {values[0]:>7.1f} {values[len(values) // 2]:>7.1f} "
        f"{values[-1]:>8.1f}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

They overlap almost completely. Tickets with an article score from 1.1 to 13.2, with 4.1 in the middle. Tickets with none score from 0.0 to 8.6, with 4.0 in the middle. No floor separates them.

A BM25 score says how well this article matches compared with the others. It doesn't say whether any article fits. A ticket about a dead mouse shares a word or two with some article, and gets a respectable score.

So this tool makes suggestions, and a person confirms them. It can't close a ticket by itself, and it shouldn't be allowed to.

</details>

### ❓ Step 5: how do I route a ticket without training anything?

By asking which solved tickets look like it. We have 150 solved tickets with their category. For a new ticket we find the 5 most similar ones, and their categories vote. That's **k nearest neighbours**, one of the oldest ideas in machine learning, and we already own the similarity: BM25.

```
new ticket:   can't join the Staff wifi on my laptop
neighbours:   network, network, hardware, network, access     → network
```

`TicketRouter` in `domain.py` does that. The votes are the BM25 scores, so a close neighbour counts for more.

First, the number to beat. The most common category among the solved tickets is `network`, and a router that always says `network` gets 8 of the 50 test tickets right.

👉 **Predict first:** how many of the 50 will the neighbours get right?

In [ ]:
from homer_ai.domain import TicketRouter


def right(router, tickets):
    return sum(router.route(t["text"])[0] == t["category"] for t in tickets)


neighbours = TicketRouter(train, k=5)
always = sum(t["category"] == neighbours.most_common for t in test)
print(f"always '{neighbours.most_common}': {always} of {len(test)}")
print(f"5 nearest tickets : {right(neighbours, test)} of {len(test)}")

ticket = next(t for t in test if "handset" in t["text"] and "Not urgent" in t["text"])
print(f"\n{ticket['category']:9} ← {ticket['text']}")
for solved, score in neighbours.neighbours(ticket["text"]):
    print(f"   {score:5.1f}  {solved['category']:9} {solved['text']}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

17 of 50. Better than always saying `network`, and still poor.

The neighbours of one failed ticket show why. The ticket is about a login. Its nearest neighbours are about a report, a colleague, a docking station, a mailbox and a monitor. What do they share with it? **Quick one:** at the start and **Not urgent.** at the end.

The search matched the greeting and the sign-off. That's **boilerplate**: words that turn up everywhere and say nothing about the problem. In a short ticket they're a big share of the text, and they pull in neighbours from every category.

</details>

### ❓ Step 6: how do I make the boilerplate stop counting?

We could keep a list of phrases to strip. But BM25 already has a tool for words that are everywhere: **idf**, the weight from 09_01 that goes to nearly 0 for a word that's in every text.

It didn't help in step 5, because among 150 separate tickets *urgent* is in only 41. That's common, but not everywhere.

So we change what a text is. Glue all solved tickets of one category into one long text. Now there are 6 texts, and *urgent*, *thanks* and *yesterday* are in all 6.

```
150 texts (one per ticket)    'urgent' in 41 of 150  → still counts
  6 texts (one per category)  'urgent' in  6 of 6    → counts for nothing
```

A new ticket is then scored against the 6 category texts, and the best one wins. That's `ProfileRouter`.

👉 **Predict first:** what will the idf of *printer* do, compared with *urgent*?

In [ ]:
from homer_ai.domain import ProfileRouter

profiles = ProfileRouter(train)
print(f"{'word':10} {'idf, 150 tickets':>17} {'idf, 6 categories':>18}")
for word in ("urgent", "thanks", "yesterday", "printer", "vpn", "password"):
    print(
        f"{word:10} {neighbours.index.idf(word):>17.2f} "
        f"{profiles.index.idf(word):>18.2f}"
    )
print(f"\none text per category: {right(profiles, test)} of {len(test)}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

In the 6-category index, *urgent*, *thanks* and *yesterday* all drop to 0.07. *printer* and *vpn* keep 1.54: each is in one category only.

And the routing goes from 17 to 24 of 50, with no list of phrases and no new code in the search. We only changed what counts as one document, and idf did the rest.

</details>

### ❓ Step 7: can the articles help with the routing too?

They should. Each article belongs to a category, and it's full of the right vocabulary: the `network` articles say *VPN*, *Wi-Fi*, *access point*, *cable*. So we add each article's text to its category's text. `ProfileRouter` takes them as a second argument.

👉 **Predict first:** 24 of 50 without the articles. And with them?

In [ ]:
with_articles = ProfileRouter(train, articles)
print(
    f"one text per category, plus its articles: "
    f"{right(with_articles, test)} of {len(test)}"
)
print()
print(f"{'category':10} {'right':>5} {'of':>4}")
for category in with_articles.categories:
    mine = [t for t in test if t["category"] == category]
    print(f"{category:10} {right(with_articles, mine):>5} {len(mine):>4}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

30 of 50 (0.60).

| Router | right, of 50 |
|---|---|
| always `network` | 8 |
| 5 nearest tickets | 17 |
| one text per category | 24 |
| … plus the articles | **30** |

By category it's uneven. `network` is 8 of 8 and `email` 8 of 9, because their tickets use words like *VPN* and *mailbox* that belong to nobody else. `software` is 3 of 10 and `security` 0 of 3: *the customer database is giving an error 500* shares no telling word with anything it has seen.

So 60% on wordings never seen before. Is that a fair picture of how it would do? The next step shows how much that depends on a choice we made before any of this ran.

</details>

### ❓ Step 8: how much does the split decide?

Our test tickets use wordings the router never saw. That's the harshest test. The usual way is gentler: shuffle all 200 tickets, train on 150, test on the other 50. Then most test tickets have a twin in the training set that differs only in its greeting.

Same data, same code, two ways to split.

👉 **Predict first:** the best router got 30 of 50 on unseen wordings. What will it get with a random split?

In [ ]:
import random

shuffled = load_tickets()
random.Random(0).shuffle(shuffled)
random_train, random_test = shuffled[:150], shuffled[150:]

print(f"{'right, of 50':28} {'split by wording':>17} {'random split':>13}")
for name, build in (
    ("5 nearest tickets", lambda solved: TicketRouter(solved, k=5)),
    ("one text per category", ProfileRouter),
    ("… plus the articles", lambda solved: ProfileRouter(solved, articles)),
):
    by_wording = right(build(train), test)
    by_chance = right(build(random_train), random_test)
    print(f"{name:28} {by_wording:>17} {by_chance:>13}")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

| right, of 50 | split by wording | random split |
|---|---|---|
| 5 nearest tickets | 17 | 39 |
| one text per category | 24 | 42 |
| … plus the articles | 30 | 42 |

The same router is 60% or 84% right, depending on how the 200 tickets were cut. Nothing about the router changed.

Which is true? Neither, quite. Real users do repeat themselves: *password expired* arrives every Monday in the same words, and a router that has seen it gets it right. Real users also find new ways to say things every day. The real number sits somewhere between, and you only learn where by testing on tickets from a later month than the ones you trained on.

It's the lesson of [05_03](../05_training_on_the_odyssey/05_03_memorising_vs_understanding.ipynb) in a new place. When somebody shows you an accuracy, the first question is how the test set was chosen.

</details>

### ❓ Step 9: would fine-tuning do better?

That's rung 3, and after [10_02](10_02_spotting_complaints.ipynb) it's a fair question. A pretrained model knows that a *handset* is a phone, which is exactly what our word search lacks.

The cell below runs the 10_02 recipe on the tickets: GPT-2, a new head with 6 classes, 150 training tickets, 15 epochs. It takes about half a minute. It only runs if GPT-2's file is already in the cache (10_02 or chapter 8 put it there); otherwise it prints one line.

👉 **Predict first:** the best word-counting router got 30 of 50. Will the fine-tuned model beat it?

In [ ]:
import os

from homer_ai.pretrained import GPT2_FILE

if os.environ.get("HOMER_SKIP_DOWNLOADS") == "1" or not GPT2_FILE.exists():
    print("GPT-2 is not in the cache (or HOMER_SKIP_DOWNLOADS=1): cell skipped")
else:
    import tiktoken
    import torch

    from homer_ai.finetune import (
        accuracy,
        encode_messages,
        make_classifier,
        train_classifier,
    )
    from homer_ai.pretrained import load_gpt2

    encoding = tiktoken.get_encoding("gpt2")
    categories = with_articles.categories

    def encode(tickets):
        ids, _ = encode_messages([("ham", t["text"]) for t in tickets], encoding, 32)
        return ids, torch.tensor([categories.index(t["category"]) for t in tickets])

    train_ids, train_labels = encode(train)
    test_ids, test_labels = encode(test)
    model = load_gpt2()
    make_classifier(model, n_classes=len(categories))
    train_classifier(
        model, train_ids, train_labels, epochs=15, lr=2e-4, report_every=95
    )
    print(f"training tickets: {accuracy(model, train_ids, train_labels):.0%} right")
    print(
        f"test tickets:     "
        f"{accuracy(model, test_ids, test_labels) * len(test):.0f} of {len(test)}"
    )

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

On my laptop (about 29 seconds of training): 96% of its training tickets right, and **26 of 50** on the test.

So no. It learned its 150 tickets almost by heart and carried little of that over to new wordings: the gap of 05_03, wide open. It lands between our second and third router, below the one that reads the articles.

I wouldn't conclude that fine-tuning is the wrong tool. 150 examples for 6 classes is very little, and I didn't tune anything. With a few thousand real tickets I'd expect it to pull ahead. What this does show is the order to work in: the word-counting router took no training, runs in milliseconds, and set a bar that half a minute of fine-tuning didn't clear. Build that first, then see what beats it.

</details>

### ❓ Step 10: what does the finished front door look like?

One function that takes a ticket and returns what a service-desk agent would see: the tray, and up to three articles to consider. We don't hide the uncertainty. The suggestions are labelled as suggestions, and a ticket the router knows nothing about says so.

In [ ]:
def triage(text):
    category, votes = with_articles.route(text)
    print(text)
    print(f"   tray: {category}" + ("" if votes else "   (a guess: no known word)"))
    for article, score in suggest_articles(text, k=3):
        print(f"   maybe: {article.id}  {article.title}  ({score:.1f})")
    print()


triage("Hi, the printer on floor 2 shows offline Thanks")
triage("got a weird email asking for my password")
triage("zzz")

<details><summary>👀 What you saw (open it after you've run the cell)</summary>

The printer ticket goes to `hardware` with the printer article first. The suspicious email goes to `access` and not `security`, because of the word *password*, but the phishing article is still among the suggestions, so the agent who opens it sees at once what it is.

And `zzz` shares no word with anything. The router falls back to the most common tray and says it's a guess, and there are no suggestions.

A person looks at every one of these. The tool saves them the first minute, and for a service desk that's most of what triage is.

</details>

## Questions learners ask

These are real questions that came up while we were building this project, with short answers.

### ❓ Why not use embeddings, so that *handset* matches *phone*?

That's the natural next step, and it's the comparison of [09_02](../09_advanced_grounded_answers/09_02_search_and_scores.ipynb): a small sentence model, or both searches merged. On tickets I'd expect it to help more than it did on Homer, because the vocabulary gap is the whole problem here. Score it on the same 26 and 50 tickets before you believe it.

### ❓ Could a big model do the routing with no examples?

Very likely, given the six category names and a line about each: that's rung 1, and for a few hundred tickets a day it's where I'd start. The router of this notebook is what you'd compare it with, and what you'd keep as the fallback when the model is unavailable.

### ❓ Why 5 neighbours?

It's a common starting value. On these tickets 1, 3, 5 and 9 neighbours give 18, 19, 17 and 16 of 50: all poor, for the reason of step 5. When changing a setting barely moves the score, the setting isn't the problem.

### ❓ Should this be an MCP tool too?

It could be: `suggest_articles` is `search_terms` with other documents, and you've built that server twice. In a service desk it more often runs inside the ticket system, when a ticket is created, with no chat involved.

## 5. Recap

- Suggesting an article is the search of chapter 9: the right one was in the first 3 for 22 of 26 test tickets. The misses are users' words that the articles don't use.
- A BM25 score can't tell you that no article fits. The tool suggests, and a person confirms.
- Routing by nearest tickets got 17 of 50, because greetings and sign-offs matched. One text per category let idf remove them (24), and adding the articles gave 30.
- The same router scored 30 of 50 on unseen wordings and 42 of 50 on a random split. Always ask how a test set was chosen.
- Half a minute of fine-tuning on 150 tickets got 26 of 50. Build the simple thing first; it's the bar.

**You can now:**

- score a suggestion tool with recall@k, computed from the ranks
- route a ticket with no training, and say why idf needs one text per category
- ask how a test set was chosen before you trust its score

**Next:** [10_04: Pitfalls, and where this goes next at work](10_04_pitfalls_and_next_steps.ipynb).